# 02 - Feature Engineering

Tujuan notebook ini:
1. Menerapkan fitur turunan domain-specific dari `src/features.py`
2. Memvalidasi apakah fitur baru punya sinyal terhadap target (pakai Information Value)
3. Menyimpan master table hasil feature engineering ke `data/processed/` untuk dipakai notebook modeling

Fungsi feature engineering sengaja ditaruh di `src/features.py` (bukan ditulis
ulang di sini) agar bisa dipakai ulang secara konsisten saat modeling
maupun saat inference nanti.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw_tables, build_master_table
from src.features import build_features
from src.config import TARGET_COL, ID_COL, PROCESSED_DATA_DIR

sns.set_style("whitegrid")


## 1. Load & Bangun Fitur

In [ ]:
tables = load_raw_tables()
master = build_master_table(tables)
df = build_features(master)

print(f"Shape sebelum feature engineering: {master.shape}")
print(f"Shape setelah feature engineering: {df.shape}")

new_cols = [c for c in df.columns if c not in master.columns]
print(f"\nFitur baru yang ditambahkan ({len(new_cols)}):")
print(new_cols)


## 2. Validasi Sinyal Fitur Baru — Information Value (IV)

Information Value adalah metric standar industri credit scoring untuk
mengukur seberapa kuat satu fitur memisahkan kelas baik vs default.
Rule of thumb industri:

| IV          | Interpretasi              |
|-------------|----------------------------|
| < 0.02      | Tidak ada predictive power |
| 0.02 - 0.1  | Lemah                      |
| 0.1 - 0.3   | Sedang                     |
| 0.3 - 0.5   | Kuat                       |
| > 0.5       | Curiga (cek data leakage)  |


In [ ]:
def calculate_iv(df: pd.DataFrame, feature: str, target: str, bins: int = 10) -> float:
    """
    Menghitung Information Value untuk satu fitur numerik kontinu,
    dengan binning kuantil sederhana.

    Catatan: ini implementasi cepat untuk eksplorasi. Untuk scorecard
    final, binning akan dilakukan lebih hati-hati pakai `optbinning`
    di notebook modeling baseline (03).
    """
    temp = df[[feature, target]].dropna().copy()
    if temp[feature].nunique() < 2:
        return np.nan

    try:
        temp["bin"] = pd.qcut(temp[feature], q=bins, duplicates="drop")
    except ValueError:
        return np.nan

    grouped = temp.groupby("bin")[target].agg(["count", "sum"])
    grouped.columns = ["total", "bad"]
    grouped["good"] = grouped["total"] - grouped["bad"]

    total_bad = grouped["bad"].sum()
    total_good = grouped["good"].sum()

    grouped["dist_bad"] = grouped["bad"] / total_bad
    grouped["dist_good"] = grouped["good"] / total_good

    # Hindari log(0) dengan epsilon kecil
    eps = 1e-6
    grouped["woe"] = np.log((grouped["dist_good"] + eps) / (grouped["dist_bad"] + eps))
    grouped["iv_contribution"] = (grouped["dist_good"] - grouped["dist_bad"]) * grouped["woe"]

    return grouped["iv_contribution"].sum()


iv_results = {}
for col in new_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        iv_results[col] = calculate_iv(df, col, TARGET_COL)

iv_df = pd.Series(iv_results).sort_values(ascending=False).to_frame("IV")
iv_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
iv_df["IV"].sort_values().plot(kind="barh", ax=ax)
ax.axvline(0.02, color="orange", linestyle="--", label="Batas 'lemah'")
ax.axvline(0.1, color="green", linestyle="--", label="Batas 'sedang'")
ax.set_title("Information Value - Fitur Hasil Feature Engineering")
ax.legend()
plt.tight_layout()
plt.show()


**Insight:** *(isi setelah melihat hasil IV — fitur mana yang punya predictive
power kuat/sedang, dan apakah ada fitur yang ternyata lemah sehingga
dipertimbangkan untuk tidak dipakai di modeling? Tulis juga jika ada fitur
dengan IV mencurigakan tinggi, perlu dicek potensi data leakage.)*

## 3. Cek Korelasi Antar Fitur Baru

Memastikan tidak ada multikolinearitas ekstrem antar fitur turunan
(misal `DEBT_TO_INCOME_RATIO` vs `ANNUITY_TO_INCOME_RATIO` bisa jadi
berkorelasi tinggi karena sama-sama melibatkan income).

In [ ]:
numeric_new_cols = [c for c in new_cols if pd.api.types.is_numeric_dtype(df[c])]
corr = df[numeric_new_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Korelasi Antar Fitur Hasil Feature Engineering")
plt.tight_layout()
plt.show()


## 4. Simpan Master Table

In [ ]:
output_path = PROCESSED_DATA_DIR / "master_table.parquet"
df.to_parquet(output_path, index=False)
print(f"Master table disimpan ke: {output_path}")
print(f"Shape final: {df.shape}")


## 5. Ringkasan & Langkah Selanjutnya

*(isi ringkasan fitur mana yang akan dibawa ke modeling, dan alasannya
berdasarkan IV + korelasi di atas)*

Lanjut ke `03_modeling_baseline.ipynb`.